# AppearanceMeetsGeometry — Tier-1 Orchestrator

Drives the full **train → segment → evaluate** pipeline for every model variant,
repeated `N_RUNS` times, to measure run-to-run spread and to carry each of the
five remaining review steps through the same machinery.

**This first run = Step 1**: retrain on the tilt-corrected normal maps and push
all variants through to ROI metrics.

How it works:
- All pipeline logic lives in the `amg_pipeline` package (the single source of
  truth, extracted from the original notebooks which now sit in `Archive/`).
- One `RunConfig` per run; **every output path is derived from a `run_id`**
  (`{channels}ch_run{N}`), so nothing ever overwrites anything.
- Every stage is **skip-if-exists**, so an interrupted sweep resumes cleanly.

Workflow below: (1) set config → (2) verify architecture → (3) preview &
check paths (no training) → (4) run sweep → (5) variance analysis.

> Authoring on macOS, heavy lifting on Windows: the only thing that changes
> between machines is the path block in the CONFIG cell. Training uses CUDA when
> available and falls back to CPU otherwise (matching the original notebooks).

## 0. Imports

In [ ]:
import os, sys
import numpy as np
import pandas as pd

# Make the amg_pipeline package importable (it lives one level up, in the repo root).
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import amg_pipeline as amg
from amg_pipeline.config import RunConfig, make_run_id
from amg_pipeline import paths

print("amg_pipeline loaded from:", os.path.dirname(amg.__file__))

## 1. CONFIG — set everything here

**This is the only cell you edit between machines / experiments.** Point the
paths at your data, set the experiment name, the variants, the run count, and
the seed. Then verify in the preview cell before running anything heavy.

In [ ]:
# === STEP 3: CLASS-IMBALANCE CONDITION (pick one) ===
# Set EXPERIMENT_NAME by hand for each run so conditions never collide and the
# frozen Step-1/2 base runs stay the control. Oversampling weight ~3.0 gives
# ~69% quarry-tile exposure per epoch (train.py prints the realized value).
CONDITION = "oversample"   # "oversample" | "dice070" | "both"
SAMPLER_WEIGHT = 3.0       # quarry-tile oversampling weight (only used when sampler is on)
_COND = {
    "oversample": dict(sampler=True,  weight=SAMPLER_WEIGHT, ce=0.5),
    "dice070":    dict(sampler=False, weight=SAMPLER_WEIGHT, ce=0.3),
    "both":       dict(sampler=True,  weight=SAMPLER_WEIGHT, ce=0.3),
}[CONDITION]

# === EXPERIMENT IDENTITY ===
EXPERIMENT_NAME = "v3_oversample_w3"            # set by hand per run
EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments")

# === SWEEP SHAPE ===
CHANNEL_VARIANTS = (3, 4, 7)   # full Table-2 fairness for Step 3; use (7,) to screen first
WIDTH = "base"                 # "slim" | "base" | "wide" — Step 3 builds on the base width
N_RUNS = 5                     # runs per variant (change to 10, etc. — no other edits needed)
SEED = 42                      # fixed this round; a later seed sweep is just changing this

# === TRAINING DATA (per the agreed setup) ===
# 3ch uses NORMALMAP_DIR + MASK_DIR; 4ch uses ORTHO_DIR + MASK_DIR; 7ch uses all three.
# 4ch is appearance-only, so its normal-map dir is irrelevant. Point each variant's
# inputs wherever you intend and CHECK in the preview cell before running.
ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_orthomosaics"
NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_normalmaps" 
MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_masks"

# === TEST WALLS ===
TEST_ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\01_test-images"
TEST_NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\02_test-normals"
TEST_MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\03_test-masks"
WALLS = ["wall1", "wall2", "wall3", "wall4"]

# Filename patterns for each wall ({wall} is substituted). Defaults match the repo.
ORTHO_PATTERN     = "{wall}_png-ortho.png"
NORMALMAP_PATTERN = "{wall}_DEM_normalmap.png"
MASK_PATTERN      = "{wall}_png-ortho.png"   # GT mask reuses the ortho name in this repo

# === HYPER-PARAMS (defaults reproduce the originals) ===
N_EPOCHS   = 300
BATCH_SIZE = 16
LR         = 1e-4

# === ROI EVAL ===
ROI_OPERATION = "closing"
KERNEL_RADIUS = 45

# Build a base config; channels/run_number are filled in per run by the sweep.
BASE_CONFIG = RunConfig(
    channels=7, run_number=1,
    experiment_name=EXPERIMENT_NAME, experiments_root=EXPERIMENTS_ROOT,
    ortho_dir=ORTHO_DIR, normalmap_dir=NORMALMAP_DIR, mask_dir=MASK_DIR,
    test_ortho_dir=TEST_ORTHO_DIR, test_normalmap_dir=TEST_NORMALMAP_DIR,
    test_mask_dir=TEST_MASK_DIR, walls=WALLS,
    ortho_pattern=ORTHO_PATTERN, normalmap_pattern=NORMALMAP_PATTERN, mask_pattern=MASK_PATTERN,
    seed=SEED, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    roi_operation=ROI_OPERATION, kernel_radius=KERNEL_RADIUS,
    width_mult=WIDTH,
    use_weighted_sampler=_COND["sampler"],
    sampler_weight=_COND["weight"],
    ce_weight=_COND["ce"],
)
print("Base config OK. Experiment:", EXPERIMENT_NAME,
      "| variants:", CHANNEL_VARIANTS, "| runs each:", N_RUNS,
      "| total runs:", len(CHANNEL_VARIANTS) * N_RUNS)

## 2. Verify the extracted model matches the originals

Because the pipeline logic was extracted by hand and the originals are archived,
this proves the model is identical before any sweep runs:
- exact parameter counts + forward-pass shapes for 3/4/7-channel;
- **loads your existing trained checkpoints** and asserts the weights map
  cleanly onto the extracted architecture (the definitive check).

If any of this fails, **stop** — do not run the sweep.

In [ ]:
amg.verify_architecture()

# Point these at your existing trained models to prove architecture identity.
# (Adjust filenames if needed; comment out any that aren't present.)
EXISTING_CKPTS = {
    3: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_3-channel_4-class-EX_300.pth"),
    4: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_4-channel_4-class-EX_300.pth"),
    7: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_7-channel_4-class-EX_300.pth"),
}
for ch, p in EXISTING_CKPTS.items():
    if os.path.exists(p):
        amg.verify_checkpoint_loads(p, channels=ch)
    else:
        print(f"(skip) no existing {ch}ch checkpoint at {p}")

## 3. Preview & check paths — NO training

Inspect-before-write gate. This derives every output path, confirms the training
directories exist and how many tiles they hold, and checks that every test-wall
input file is present. **Fix any ✗ here before running the sweep.**

In [ ]:
import glob, dataclasses

def _count_pngs(d):
    return len(glob.glob(os.path.join(d, "*.png"))) if os.path.isdir(d) else None

print("=== TRAINING DIRECTORIES ===")
for label, d in [("ortho", ORTHO_DIR), ("normalmap", NORMALMAP_DIR), ("mask", MASK_DIR)]:
    n = _count_pngs(d)
    print(f"  {'OK ' if n else '✗  '} {label:10s} {d}  ->  {n if n is not None else 'MISSING'} png")

print("\n=== TEST WALL INPUTS ===")
all_ok = True
for wall in WALLS:
    for label, fn in [("ortho", paths.test_ortho_path(BASE_CONFIG, wall)),
                      ("normal", paths.test_normalmap_path(BASE_CONFIG, wall)),
                      ("mask", paths.test_mask_path(BASE_CONFIG, wall))]:
        exists = os.path.exists(fn)
        all_ok = all_ok and exists
        print(f"  {'OK ' if exists else '✗  '} {wall} {label:7s} {fn}")

print("\n=== DERIVED OUTPUT PATHS (sample: 7ch_run1) ===")
sample = dataclasses.replace(BASE_CONFIG, channels=7, run_number=1)
print("  checkpoint :", paths.checkpoint_path(sample))
print("  config.json:", paths.config_json_path(sample))
print("  seg (wall1):", paths.segmentation_raw_path(sample, "wall1"))
print("  metrics dir:", paths.metrics_wall_dir(sample, "wall1"))
print("  manifest   :", paths.manifest_path(sample))

print("\n=== RUN IDS TO BE PRODUCED ===")
for cfg in amg.build_configs(BASE_CONFIG, CHANNEL_VARIANTS, N_RUNS):
    print("   ", make_run_id(cfg))

print("\nAll test inputs present." if all_ok else "\n*** Some inputs MISSING — fix before running. ***")

## 4. Run the sweep

Gated by `DO_RUN` so it can't fire by accident. Each run does
train → segment(all walls) → evaluate(all walls) → append manifest, and every
stage skips work already on disk, so re-running resumes an interrupted sweep.

Stage gates (`DO_TRAIN/DO_SEGMENT/DO_EVALUATE`) let you run stages separately —
e.g. recompute metrics without retraining by setting `DO_TRAIN=DO_SEGMENT=False`.

In [ ]:
DO_RUN = True        # <-- set True to launch the sweep
DO_TRAIN = True
DO_SEGMENT = True
DO_EVALUATE = True
FORCE = False         # True ignores skip-if-exists and recomputes everything

if DO_RUN:
    amg.run_sweep(BASE_CONFIG, channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS,
                  do_train=DO_TRAIN, do_segment=DO_SEGMENT, do_evaluate=DO_EVALUATE,
                  force=FORCE)
else:
    print("DO_RUN is False — set it to True to launch. "
          f"Would run {len(CHANNEL_VARIANTS) * N_RUNS} configs.")

## 5. Variance analysis

Loads the master manifest and reports mean ± std (and min/max) of IoU/F1 across
the `N_RUNS` repeats, per variant per wall. This is the spread the whole exercise
is about — it tells you whether a single-run comparison between variants is
trustworthy or within the noise band. Remember: with the seed fixed, this is
run-time/implementation variance (init + split held fixed).

In [ ]:
mf_path = paths.manifest_path(BASE_CONFIG)
if not os.path.exists(mf_path):
    print("No manifest yet — run the sweep first:", mf_path)
else:
    mf = pd.read_csv(mf_path)
    metrics = ["IoU_mean_stones", "IoU_Ashlar", "IoU_Polygonal", "IoU_Quarry", "macro_F1"]
    agg = (mf.groupby(["channels", "wall"])[metrics]
             .agg(["mean", "std", "min", "max"]))
    pd.set_option("display.width", 200, "display.max_columns", 50)

    print("=== SPREAD ACROSS RUNS (AllWalls aggregate) ===")
    allw = agg.xs("AllWalls", level="wall")
    for ch in sorted(mf["channels"].unique()):
        row = allw.loc[ch]
        m, s = row[("IoU_mean_stones", "mean")], row[("IoU_mean_stones", "std")]
        lo, hi = row[("IoU_mean_stones", "min")], row[("IoU_mean_stones", "max")]
        print(f"  {ch}ch  mean-stone IoU = {m:.4f} ± {s:.4f}  (min {lo:.4f}, max {hi:.4f})")

    print("\n=== FULL TABLE (mean ± std per variant per wall) ===")
    display(agg.round(4))

    n_per = mf[mf.wall == "AllWalls"].groupby("channels").size()
    print("\nRuns completed per variant:\n", n_per.to_string())